In [1]:
import fitsio
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
from astropy.table import Table
data_loc = "/home/rohan/agn_photometry/data/dr16q_prop_May01_2024.fits"
columns = [
    'FEII_OPT_EW', 'FEII_OPT_EW_ERR', 'HBETA_BR', 'HBETA_BR_ERR', 
    'Z_FIT', 'LOGL5100', 'RA', 'DEC', 'SDSS_NAME', 'PLATE', 'MJD', 'FIBERID', 'OBJID'
]
raw_data = fitsio.read(data_loc, ext=1, columns=columns)
df = pl.DataFrame({col: raw_data[col].tolist() for col in raw_data.dtype.names}).lazy()

In [2]:
processed_df = (
    df.with_columns([
        pl.col("HBETA_BR").list.get(5).alias("hb_ew"),
        pl.col("HBETA_BR").list.get(4).alias("fwhm_hbeta"),
        pl.col("HBETA_BR_ERR").list.get(5).alias("hb_ew_err"),
        pl.col("HBETA_BR_ERR").list.get(4).alias("fwhm_hbeta_err"),
    ])
    .with_columns([
        (pl.col("FEII_OPT_EW") / pl.col("hb_ew")).alias("R_Fe"),
        (((pl.col("FEII_OPT_EW_ERR") / pl.col("FEII_OPT_EW")).pow(2) + 
          (pl.col("hb_ew_err") / pl.col("hb_ew")).pow(2)).sqrt()
        ).fill_nan(1.0).alias("rel_err_rfe"),
        (pl.col("fwhm_hbeta_err") / pl.col("fwhm_hbeta")).fill_nan(1.0).alias("rel_err_fwhm"),
    ])
    .filter(
        (pl.col("R_Fe") != 0) & 
        (pl.col("fwhm_hbeta") != 0) & 
        (pl.col("rel_err_rfe") < 0.20) &
        (pl.col("rel_err_fwhm") < 0.20) &
        (pl.col("Z_FIT") > 0.05) &
        (pl.col("Z_FIT") < 0.6) &
        (pl.col("hb_ew") > 10) &          # broad Hβ EW > 10Å — stars won't have this
        (pl.col("FEII_OPT_EW") > 5) &     # real optical FeII emission
        (pl.col("fwhm_hbeta") > 1000)     # FWHM > 1000 km/s — stellar absorption is narrow
    )
    .collect()
)

In [3]:
R_Fe_filtered = processed_df["R_Fe"].to_numpy()
fwhm_filtered = processed_df["fwhm_hbeta"].to_numpy()
z_fit_filtered = processed_df["Z_FIT"].to_numpy()
logl5100_filtered = processed_df["LOGL5100"].to_numpy()

In [4]:
print(f"Total High-Quality Sources Isolated: {len(R_Fe_filtered)}")

Total High-Quality Sources Isolated: 12224


In [5]:
import requests

def get_classes_batch(rows):
    url = "https://skyserver.sdss.org/dr18/SkyServerWS/SearchTools/SqlSearch"
    conditions = " OR ".join(
        f"(s.plate={r['PLATE']} AND s.mjd={r['MJD']} AND s.fiberid={r['FIBERID']})"
        for r in rows
    )
    query = f"""SELECT s.plate, s.mjd, s.fiberid, s.class, s.subClass, s.zWarning, p.type
FROM SpecObj s JOIN PhotoObj p ON s.bestobjid = p.objid
WHERE {conditions}"""
    try:
        r = requests.get(url, params={"cmd": query, "format": "json"}, timeout=30)
        r.raise_for_status()
        res = r.json()
        result_map = {}
        if res and isinstance(res, list):
            for row in res[0].get("Rows", []):
                key = (row["plate"], row["mjd"], row["fiberid"])
                result_map[key] = {
                    "class": str(row.get("class", "")).strip().upper(),
                    "subClass": str(row.get("subClass", "")).strip().upper(),
                    "zWarning": int(row.get("zWarning", 1)),
                    "type": int(row.get("type", 6))
                }
        return result_map
    except Exception as e:
        print(f"[WARNING] Batch API call failed: {e}")
        return {}

def get_confirmed_agns(candidates_df, quota=20, batch_size=50):
    confirmed = []
    all_rows = list(candidates_df.iter_rows(named=True))
    for i in range(0, len(all_rows), batch_size):
        if len(confirmed) >= quota:
            break
        batch = all_rows[i:i+batch_size]
        class_map = get_classes_batch(batch)
        for row in batch:
            if len(confirmed) >= quota:
                break
            key = (row["PLATE"], row["MJD"], row["FIBERID"])
            result = class_map.get(key)
            if result is None:
                print(f"  Unverified (not in SpecObj): {key}")
                continue
            obj_class = result["class"]
            subclass = result["subClass"]
            z_warn = result["zWarning"]
            phot_type = result["type"]  # 6=STAR, 3=GALAXY in PhotoObj
            if z_warn != 0:
                print(f"  Rejected zWarning={z_warn}: {key}")
                continue
            if obj_class == "STAR" or phot_type == 6:
                print(f"  Rejected STAR (spec={obj_class}, photo={phot_type}): {key}")
                continue
            if obj_class == "QSO":
                confirmed.append(row)
            elif obj_class == "GALAXY" and ("AGN" in subclass or "BROADLINE" in subclass):
                confirmed.append(row)
            else:
                print(f"  Rejected ({obj_class} / {subclass}): {key}")
    return pl.DataFrame(confirmed)

In [6]:
TARGET_QUOTA = 20

top_10_rfe = get_confirmed_agns(
    processed_df.filter(pl.col("R_Fe") > 3).sort("LOGL5100", descending=True),
    quota=TARGET_QUOTA
)
print(f"High R_Fe confirmed AGNs: {len(top_10_rfe)}")

top_10_fwhm = get_confirmed_agns(
    processed_df.filter(pl.col("fwhm_hbeta") > 12000).sort("LOGL5100", descending=True),
    quota=TARGET_QUOTA
)
print(f"High FWHM confirmed AGNs: {len(top_10_fwhm)}")

  Unverified (not in SpecObj): (11379, 58438, 592)
  Rejected STAR (spec=QSO, photo=6): (9363, 57742, 730)
  Rejected STAR (spec=QSO, photo=6): (8210, 57426, 734)
  Rejected STAR (spec=QSO, photo=6): (1690, 53475, 4)
  Rejected STAR (spec=QSO, photo=6): (1797, 54507, 220)
  Rejected STAR (spec=QSO, photo=6): (7600, 56984, 472)
  Rejected STAR (spec=QSO, photo=6): (3836, 55302, 726)
  Rejected STAR (spec=QSO, photo=6): (7871, 56902, 74)
  Unverified (not in SpecObj): (623, 52051, 509)
  Rejected STAR (spec=QSO, photo=6): (939, 52636, 456)
  Rejected STAR (spec=QSO, photo=6): (1467, 53115, 471)
  Rejected STAR (spec=QSO, photo=6): (11044, 58508, 838)
  Rejected STAR (spec=QSO, photo=6): (7595, 56957, 190)
  Rejected STAR (spec=QSO, photo=6): (8853, 57459, 232)
  Rejected STAR (spec=QSO, photo=6): (9561, 57809, 230)
  Rejected STAR (spec=QSO, photo=6): (726, 52226, 613)
  Rejected STAR (spec=QSO, photo=6): (10915, 58257, 428)
  Rejected STAR (spec=QSO, photo=6): (10901, 58397, 175)
  Reje

In [7]:
combined_targets = pl.concat([top_10_rfe, top_10_fwhm]).unique()
print(f"Total Unique Extremity Targets Combined: {len(combined_targets)}")

Total Unique Extremity Targets Combined: 32


In [8]:
combined_targets = combined_targets.with_columns([
    (pl.lit("https://skyserver.sdss.org/dr18/VisualTools/explore/summary?plate=") +
     pl.col("PLATE").cast(pl.Utf8) + pl.lit("&mjd=") + pl.col("MJD").cast(pl.Utf8) +
     pl.lit("&fiber=") + pl.col("FIBERID").cast(pl.Utf8)).alias("Link")
])

In [9]:
combined_targets.head

<bound method DataFrame.head of shape: (32, 21)
┌──────────────┬───────┬───────┬─────────┬───┬──────────┬─────────────┬──────────────┬─────────────┐
│ SDSS_NAME    ┆ PLATE ┆ MJD   ┆ FIBERID ┆ … ┆ R_Fe     ┆ rel_err_rfe ┆ rel_err_fwhm ┆ Link        │
│ ---          ┆ ---   ┆ ---   ┆ ---     ┆   ┆ ---      ┆ ---         ┆ ---          ┆ ---         │
│ str          ┆ i64   ┆ i64   ┆ i64     ┆   ┆ f64      ┆ f64         ┆ f64          ┆ str         │
╞══════════════╪═══════╪═══════╪═════════╪═══╪══════════╪═════════════╪══════════════╪═════════════╡
│ 152942.19+35 ┆ 10742 ┆ 58198 ┆ 490     ┆ … ┆ 0.169517 ┆ 0.160917    ┆ 0.11823      ┆ https://sky │
│ 0851.3       ┆       ┆       ┆         ┆   ┆          ┆             ┆              ┆ server.sdss │
│              ┆       ┆       ┆         ┆   ┆          ┆             ┆              ┆ .org/dr1…   │
│ 075005.28+29 ┆ 1060  ┆ 52636 ┆ 280     ┆ … ┆ 4.077647 ┆ 0.117745    ┆ 0.109354     ┆ https://sky │
│ 2944.3       ┆       ┆       ┆         ┆ 

In [10]:
final_target_table = combined_targets.select([
    pl.col("SDSS_NAME").cast(pl.Utf8).alias("Name of the source"),
    pl.col("RA"),
    pl.col("DEC"),
    pl.col("Z_FIT").alias("redshift"),
    pl.col("FIBERID").alias("FIBER"),
    pl.col("MJD"),
    pl.col("PLATE"),
    pl.col("Link")
])

In [11]:
output_path = "/home/rohan/agn_photometry/photometry/results/Target_Table.csv"
final_target_table.write_csv(output_path)
print(f"Target table successfully generated with {final_target_table.height} unique sources.")

Target table successfully generated with 32 unique sources.


In [12]:
from astropy.table import Table
display_targets = Table({col: final_target_table[col].to_numpy() for col in final_target_table.columns})

display_targets.pprint_all()

Name of the source         RA                 DEC              redshift      FIBER  MJD  PLATE                                             Link                                           
------------------ ------------------ ------------------- ------------------ ----- ----- ----- -------------------------------------------------------------------------------------------
152942.19+350851.3  232.4258318203353   35.14758608779926              0.287   490 58198 10742 https://skyserver.sdss.org/dr18/VisualTools/explore/summary?plate=10742&mjd=58198&fiber=490
075005.28+292944.3         117.522012           29.495662           0.327794   280 52636  1060  https://skyserver.sdss.org/dr18/VisualTools/explore/summary?plate=1060&mjd=52636&fiber=280
095919.14+090659.4         149.829764            9.116513           0.560051   210 52999  1307  https://skyserver.sdss.org/dr18/VisualTools/explore/summary?plate=1307&mjd=52999&fiber=210
150348.33+193941.3         225.951406           19.661489        